In [2]:
# Install dependencies (uncomment if running on Colab / fresh environment)
!pip install sentence-transformers faiss-cpu pypdf transformers torch -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 26.1 MB/s eta 0:00:00


In [3]:
import os
import re
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

In [6]:
with open("sample_document.txt", "w") as f:
    f.write("""RAG stands for Retrieval-Augmented Generation. It combines a retriever with a generator to answer questions using external documents.
The capital of France is Paris. Paris is known for the Eiffel Tower and the Louvre Museum.""")

In [7]:
def load_document(path: str) -> str:
    """Load a .txt or .pdf file and return its raw text."""
    if path.lower().endswith(".pdf"):
        from pypdf import PdfReader
        reader = PdfReader(path)
        return "\n".join(page.extract_text() or "" for page in reader.pages)
    else:
        with open(path, "r", encoding="utf-8") as f:
            return f.read()

DOC_PATH = "sample_document.txt"  # <-- change this to your own PDF/TXT file
raw_text = load_document(DOC_PATH)
print(f"Loaded document: {len(raw_text)} characters")

Loaded document: 224 characters


In [8]:
def chunk_text(text: str, chunk_size: int = 500, overlap: int = 100) -> list:
    """Split text into overlapping chunks (by characters)."""
    text = re.sub(r"\s+", " ", text).strip()
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return [c for c in chunks if c.strip()]

chunks = chunk_text(raw_text)
print(f"Created {len(chunks)} chunks")
print("\nSample chunk:\n", chunks[0][:300] if chunks else "No chunks created")

Created 1 chunks

Sample chunk:
 RAG stands for Retrieval-Augmented Generation. It combines a retriever with a generator to answer questions using external documents. The capital of France is Paris. Paris is known for the Eiffel Tower and the Louvre Museum.


In [9]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(chunks, convert_to_numpy=True, normalize_embeddings=True)
print("Embedding matrix shape:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding matrix shape: (1, 384)


In [10]:
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)   # normalized embeddings + inner product = cosine similarity
index.add(embeddings)
print(f"FAISS index built with {index.ntotal} vectors")

FAISS index built with 1 vectors


In [11]:
def retrieve(query: str, top_k: int = 3) -> list:
    q_emb = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    scores, indices = index.search(q_emb, top_k)
    return [chunks[i] for i in indices[0] if i != -1]

# Quick test
test_results = retrieve("What is this document about?", top_k=2)
for i, r in enumerate(test_results):
    print(f"--- Retrieved chunk {i+1} ---\n{r[:200]}\n")

--- Retrieved chunk 1 ---
RAG stands for Retrieval-Augmented Generation. It combines a retriever with a generator to answer questions using external documents. The capital of France is Paris. Paris is known for the Eiffel Towe



In [12]:
def build_prompt(question: str, context_chunks: list) -> str:
    context = "\n\n".join(f"[Chunk {i+1}]\n{c}" for i, c in enumerate(context_chunks))
    return f"""Answer the question using ONLY the context below.
If the answer isn't in the context, say "I don't know based on the given document."

Context:
{context}

Question: {question}

Answer:"""

sample_prompt = build_prompt("What is this document about?", test_results)
print(sample_prompt)

Answer the question using ONLY the context below.
If the answer isn't in the context, say "I don't know based on the given document."

Context:
[Chunk 1]
RAG stands for Retrieval-Augmented Generation. It combines a retriever with a generator to answer questions using external documents. The capital of France is Paris. Paris is known for the Eiffel Tower and the Louvre Museum.

Question: What is this document about?

Answer:


In [15]:
# Clear any corrupted cached model files
!rm -rf ~/.cache/huggingface

# Upgrade to compatible versions
!pip install -U transformers sentencepiece accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 28.1 MB/s eta 0:00:00


In [17]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def generate_answer_local(prompt: str) -> str:
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    outputs = model.generate(**inputs, max_new_tokens=200)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [18]:
def ask(question: str, top_k: int = 3, use_openai: bool = False) -> str:
    relevant_chunks = retrieve(question, top_k=top_k)
    prompt = build_prompt(question, relevant_chunks)
    if use_openai:
        return generate_answer_openai(prompt)   # requires Option B uncommented
    return generate_answer_local(prompt)

In [19]:
question = "What is this document about?"
answer = ask(question)
print("Q:", question)
print("A:", answer)

Q: What is this document about?
A: RAG


In [20]:
question = "Type your question here"
answer = ask(question)
print("Q:", question)
print("A:", answer)

Q: Type your question here
A: I don't know based on the given document.
